# 2.2 多头注意力与参数量 / Multi-head attention and parameter counts

把一次注意力拆成并行的若干个头。实现它，并亲手复现 GPT-2 的注意力层。

Split one attention into several parallel heads. Implement it and reproduce GPT-2's attention layer yourself.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 实现多头注意力并与 PyTorch 对比 / Implement multi-head attention and compare with PyTorch

补全 `MultiHeadAttention.forward`，重点是五步形状变换。测试会把 `nn.MultiheadAttention` 的权重复制到你的模块里，在头数为 1、4、8 时比较输出和每个头的注意力权重，并验证参数量与头数无关。

Complete `MultiHeadAttention.forward`, where the five shape changes are the main point. The tests copy the weights of `nn.MultiheadAttention` into your module, compare the output and the per-head attention weights for 1, 4 and 8 heads, and verify that the parameter count does not depend on the number of heads.

In [ ]:
import math
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    """Multi-head self-attention.

    self.qkv  maps d_model -> 3 * d_model  (queries, keys, values stacked in that order)
    self.proj maps d_model -> d_model      (the output projection W_O)
    """

    def __init__(self, d_model, n_head):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head, self.d_head = n_head, d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        """x: (B, T, d_model).  mask: bool (T, T), True = may attend.
        Returns (output (B, T, d_model), weights (B, n_head, T, T))."""
        B, T, d = x.shape
        # TODO 1: q, k, v = self.qkv(x) split into three chunks of size d      -> each (B, T, d)
        # TODO 2: reshape each to (B, T, n_head, d_head), then swap to (B, n_head, T, d_head)
        # TODO 3: scores (B, n_head, T, T), scaled by sqrt(d_head); apply the mask; softmax
        # TODO 4: weighted sum of values (B, n_head, T, d_head)
        # TODO 5: swap back to (B, T, n_head, d_head), merge to (B, T, d), apply self.proj
        raise NotImplementedError


def mha_param_count(d_model, n_head):
    """Number of parameters of the module above (with biases)."""
    # TODO 6: does n_head appear in the answer?
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
import torch.nn as nn

torch.manual_seed(0)
B, T, d = 2, 6, 64
x = torch.randn(B, T, d)

for h in (1, 4, 8):
    ref = nn.MultiheadAttention(d, h, batch_first=True)
    mine = MultiHeadAttention(d, h)
    with torch.no_grad():                              # same weights in both modules
        mine.qkv.weight.copy_(ref.in_proj_weight); mine.qkv.bias.copy_(ref.in_proj_bias)
        mine.proj.weight.copy_(ref.out_proj.weight); mine.proj.bias.copy_(ref.out_proj.bias)

    out, w = mine(x)
    ref_out, ref_w = ref(x, x, x, need_weights=True, average_attn_weights=False)
    assert out.shape == (B, T, d) and w.shape == (B, h, T, T), f"n_head={h}: wrong shapes"
    assert torch.allclose(out, ref_out, atol=1e-5), f"n_head={h}: output differs from nn.MultiheadAttention"
    assert torch.allclose(w, ref_w, atol=1e-5), f"n_head={h}: attention weights differ"

    causal = torch.tril(torch.ones(T, T, dtype=torch.bool))            # True = may attend
    out_c, w_c = mine(x, mask=causal)
    # note: nn.MultiheadAttention uses the opposite convention, True = NOT allowed
    ref_c, _ = ref(x, x, x, attn_mask=~causal, need_weights=False)
    assert torch.allclose(out_c, ref_c, atol=1e-5), f"n_head={h}: masked output differs"
    assert (w_c[..., ~causal] == 0).all(), "no weight may fall on future positions"

    n_params = sum(p.numel() for p in ref.parameters())
    assert mha_param_count(d, h) == n_params == sum(p.numel() for p in mine.parameters())

assert mha_param_count(768, 12) == 2_362_368, "one GPT-2 small attention layer has 2,362,368 parameters"
print("m22_mha: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

`self.qkv(x).chunk(3, dim=-1)` 按 query、key、value 的顺序得到三个 `(B, T, d)` 的张量。

`self.qkv(x).chunk(3, dim=-1)` gives three tensors of shape `(B, T, d)` in the order query, key, value.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

拆头是 `t.view(B, T, n_head, d_head).transpose(1, 2)`。不能直接 `view(B, n_head, T, d_head)`，那样会把不同位置的数据混在一起。

Split heads with `t.view(B, T, n_head, d_head).transpose(1, 2)`. A direct `view(B, n_head, T, d_head)` would mix data from different positions.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

合并时先 `transpose(1, 2)` 换回 `(B, T, n_head, d_head)`，再 `.contiguous().view(B, T, d)`。缩放用的是 `d_head` 的平方根。

To merge, first `transpose(1, 2)` back to `(B, T, n_head, d_head)`, then `.contiguous().view(B, T, d)`. The scaling uses the square root of `d_head`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head, self.d_head = n_head, d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)                                   # each (B, T, d)
        split = lambda t: t.view(B, T, self.n_head, self.d_head).transpose(1, 2)  # (B, h, T, d_head)
        q, k, v = split(q), split(k), split(v)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)                # (B, h, T, T)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        out = weights @ v                                                        # (B, h, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, d)                     # glue the heads back together
        return self.proj(out), weights


def mha_param_count(d_model, n_head):
    return (3 * d_model * d_model + 3 * d_model) + (d_model * d_model + d_model)  # independent of n_head
```

</details>

## 2. 用权重复现 GPT-2 的注意力层 / Reproduce GPT-2's attention layer from its weights

只用 GPT-2 某一层注意力的两组权重 `c_attn` 和 `c_proj`，写出完整的前向计算，包括拆头、因果掩码和合并。测试会在第 0、5、11 层上与模型自带的注意力层对比。完成后，阶段 0 流程图中“12 层”里的一半已经由你亲手实现。

Using only the two weight sets `c_attn` and `c_proj` of one GPT-2 attention layer, write the full forward pass including head splitting, causal mask and merging. The tests compare with the model's own attention layer at layers 0, 5 and 11. Once done, you have implemented half of the 12 layers box of the stage 0 diagram yourself.

需要联网下载模型。/ Downloads a model.

In [ ]:
import math
import torch
from transformers import AutoModelForCausalLM

# One attention layer of GPT-2 small is  attn = model.transformer.h[i].attn  with
#   attn.c_attn  the combined Q, K, V projection.  weight (768, 2304), bias (2304,)
#   attn.c_proj  the output projection.            weight (768, 768),  bias (768,)
# These are "Conv1D" layers, which compute  y = x @ weight + bias
# (note: no transpose, unlike nn.Linear).  GPT-2 small has 12 heads of size 64.


@torch.no_grad()
def gpt2_attention(x, attn, n_head=12):
    """Recompute attn(x) from its weights. x: (B, T, 768). Returns (B, T, 768).

    GPT-2 is causal: position i may only attend to positions <= i.
    """
    B, T, d = x.shape
    # TODO 1: qkv = x @ attn.c_attn.weight + attn.c_attn.bias, split into q, k, v
    # TODO 2: split into heads, compute causal scaled dot-product attention
    # TODO 3: merge the heads, apply c_proj
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM

gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").eval()
torch.manual_seed(0)
x = torch.randn(2, 9, 768)
for layer in (0, 5, 11):
    attn = gpt2.transformer.h[layer].attn
    with torch.no_grad():
        ref = attn(x)
        ref = ref[0] if isinstance(ref, tuple) else ref
    out = gpt2_attention(x, attn)
    assert out.shape == ref.shape
    assert torch.allclose(out, ref, atol=1e-3), f"layer {layer}: max difference {(out - ref).abs().max():.4f}"
print("m22_gpt2attn: all tests passed. You have re-implemented GPT-2's attention layer.")

<details><summary><b>提示 1 / Hint 1</b></summary>

GPT-2 的 `Conv1D` 层计算的是 `x @ weight + bias`，权重形状是 `(输入, 输出)`，不需要转置。

GPT-2's `Conv1D` layers compute `x @ weight + bias` with weights of shape `(in, out)`, so no transpose is needed.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`(x @ attn.c_attn.weight + attn.c_attn.bias).split(768, dim=-1)` 得到 q、k、v。之后的步骤与上一题相同，再加上 `torch.tril` 生成的因果掩码。

`(x @ attn.c_attn.weight + attn.c_attn.bias).split(768, dim=-1)` gives q, k and v. The rest follows the previous exercise, plus a causal mask from `torch.tril`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch
from transformers import AutoModelForCausalLM


@torch.no_grad()
def gpt2_attention(x, attn, n_head=12):
    B, T, d = x.shape
    q, k, v = (x @ attn.c_attn.weight + attn.c_attn.bias).split(d, dim=-1)
    split = lambda t: t.view(B, T, n_head, d // n_head).transpose(1, 2)        # (B, h, T, 64)
    q, k, v = split(q), split(k), split(v)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d // n_head)
    causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
    scores = scores.masked_fill(~causal, float("-inf"))
    out = torch.softmax(scores, dim=-1) @ v
    out = out.transpose(1, 2).contiguous().view(B, T, d)
    return out @ attn.c_proj.weight + attn.c_proj.bias
```

</details>